# Step 1. Framing the Problem

Some students leave higher education before finishing their degree. A school that spots who is at risk early, around the time a student enrolls, can offer support before that student drops out.

This notebook sets up the problem, so the rest of the project has a solid start. It answers four questions.

1. **What is the real problem, and who does it help?**
2. **What kind of machine learning task is this, and why that kind?**
3. **How is success measured, in model terms and in money terms?**
4. **What data does the model use, and what is left out, and why?**

Each answer carries into later steps. The choices here shape how the data is prepared in Step 3 and how the model is checked for fairness in Step 5.

In [1]:
import sys
from pathlib import Path

# I locate the repo root once, so src/ becomes importable from notebooks/.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from src.paths import RAW_DIR

In [2]:
# RAW_DIR comes from src.paths and resolves no matter where I launch the
# notebook from. The dataset uses ';' as its separator, not ',', so sep=';'
# is required.
df = pd.read_csv(RAW_DIR / "dropout.csv", sep=";")

# I check what loaded. Table size, and how the outcome splits.
print("rows, columns", df.shape)
print()
print(df["Target"].value_counts())

rows, columns (4424, 37)

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


## 1. The Problem, and Who It Helps

Every year, some students leave higher education before finishing their degree. The cost falls on three sides. The student loses time, money, and future earnings. The institution loses tuition and its investment in that student. Society loses part of the skilled workforce it helped train.

The data comes from the Polytechnic Institute of Portalegre (IPP), a public college in Portugal. IPP uses this data in a tool that helps its tutoring team find students at risk of leaving early, so support can reach them in time.

This project mirrors that use. The model looks only at what is known when a student enrolls, and estimates how likely that student is to drop out. It serves the tutoring staff, who cannot watch every student closely and need to focus limited attention on those most at risk. The model turns a long list of students into a short ranked one, the students worth contacting first.

The model answers one question. Which students should the support team contact early? A correct flag reaches a student who would otherwise slip away. A wrong one wastes staff time on a student who was fine, or misses a student who needed help.

## 2. Scope and Population

The data covers students at one institution, IPP, across 17 undergraduate degrees, from the 2008 to 2009 year through 2018 to 2019. Every student sits inside the Portuguese system, enrolled in a public college, under its rules on tuition, scholarships, and admissions.

This sets a clear boundary. The model applies to an IPP style Portuguese polytechnic. It is not a general dropout predictor for any school in any country. A model trained on one institution learns that institution's patterns, its student mix, its local economy, its years. Those patterns do not carry cleanly to a different school elsewhere.

The method carries even though the model does not. The steps I take here, framing the problem, auditing the data for bias, comparing models, checking fairness, apply to any school with similar data. This honesty shapes every claim I make, and it returns in Step 5 as a stated limit on how far the results reach.

## 3. What the Model Sees, and What It Leaves Out

The dataset holds two kinds of information. The first is known when a student enrolls, including age, admission grade, prior schooling, scholarship status, and family background. The second is recorded later, during the first and second semesters, including how many courses the student took, passed, and failed, and the grades earned.

The second kind stays out on purpose. Semester performance predicts dropout almost perfectly, because a student failing courses is a student already leaving. A model built on it would score high and help no one, since it only names a dropout once the dropout is under way. The tool exists to warn before that stage, so anything recorded after enrollment is excluded.

This is a deliberate trade. The model trains on enrollment time features only, so its raw accuracy is lower than a model that reads semester results. I accept the lower accuracy here. An early warning that is somewhat right beats a late confirmation that is nearly certain, because only the early warning leaves time to act.

The name for the excluded features is leakage, information from the outcome slipping into the inputs. The same choice returns in Step 3, where the semester columns are dropped in code, and in Step 5, where it becomes an accepted limit on accuracy.

## 4. The Task, and Why This Kind

The model sorts each student into one of two groups. Likely to drop out, or likely to graduate. This is a binary classification task.

The original data has three outcomes. Dropout, Graduate, and Enrolled. Enrolled means the student was still studying when the data was recorded, so the final outcome is not yet known. A student with no settled outcome cannot teach the model what dropout looks like, so the Enrolled rows come out. What remains is two settled outcomes, which fits a yes or no question.

Other task types do not fit. Regression predicts a number, but the answer is a category, not a quantity. Clustering groups students without known labels, but the labels are known here, dropout or graduate. Recommendation suggests items to a user, which is a different problem. The outcome is a resolved category with a clear label, so classification is the right choice.

In [3]:
# The task is binary, Dropout vs Graduate. Enrolled has no settled outcome, so I drop it.
binary = df[df["Target"].isin(["Dropout", "Graduate"])].copy()
binary["dropout"] = (binary["Target"] == "Dropout").astype(int)

n = len(binary)
rate = binary["dropout"].mean()

print("rows after dropping Enrolled", n)
print("dropout rate", round(rate * 100, 1), "percent")
print()
print(binary["Target"].value_counts())

rows after dropping Enrolled 3630
dropout rate 39.1 percent

Target
Graduate    2209
Dropout     1421
Name: count, dtype: int64


## 5. How Success Is Measured

Two kinds of measure matter here. How well the model works, and what it is worth.

The model outputs a probability, not a hard label. It says a student is 0.82 likely to drop out, not simply dropout. Turning that into a decision needs a cutoff, and **the cutoff is a choice, not a given**. Move it and every count changes. So the measures that judge the model have to be the ones that hold across every cutoff at once, and the cutoff itself gets picked separately, on purpose, in Step 4. AUC means area under the curve.

Accuracy is the obvious measure, and it misleads here. About 39 in 100 students in the two outcome groups drop out, shown above. A lazy model that calls everyone a graduate would be right about 61 percent of the time while catching no one at risk. High accuracy, no value. So I report accuracy but I do not trust it as the headline.

Two measures pick the model, because neither depends on where the cutoff sits.

PR AUC, from precision and recall. Precision asks, of the students I flag, how many truly drop out. PR AUC pairs that with recall across every cutoff. It ignores the graduates I correctly leave alone, so it stays honest when the two groups are uneven, which they are here. This is the one I tune on.

ROC AUC. I take one random dropout and one random graduate. This is the chance the model gives the dropout the higher risk score. It reads as a ranking score, 1.0 for perfect order and 0.5 for a coin flip. It matches the real use, a ranked list staff work down.

One measure judges the deployed system, once the cutoff exists.

Recall on the dropout group. Of the students who truly drop out, how many did the model flag? A missed student is the costly error, since that student loses the help the tool exists to provide. This is what the school actually cares about, and it is the number I would report to them.

But recall cannot pick a model, and it is worth being clear about why. **Recall is not a property of a model. It is a property of a model and a cutoff.** Quote a recall figure without saying where the cutoff sits and you have said almost nothing, because I can raise recall to 1.0 on any model by flagging every student. So PR AUC and ROC AUC choose the model, Step 4 chooses the cutoff using the value numbers in the next section, and only then does recall mean something.

## 6. The Business Value

The model earns its place by saving more than it costs. The value has a simple shape. Students kept, minus the cost of reaching out to flagged students.

I set three assumptions, stated plainly so the estimate stays honest.

A kept student is worth about 2,100 euros. This follows the Portuguese public tuition cap, roughly 697 euros a year over about three years. I set it as a deliberate low estimate. Tuition is a direct, citable figure, while the wider loss to the student and society is larger but harder to defend.

Reaching out does not guarantee keeping. A flag starts a conversation. It does not keep the student on its own. So I treat 2,100 euros as an upper bound for each caught student, unless I scale it by how often outreach actually works, for example keeping 3 in 10 of the at risk students reached.

Outreach cost is an estimate. I report it across a range, roughly 50, 150, or 300 euros per flagged student, since the real figure depends on the school.

This keeps the value honest. It rewards catching at risk students, charges for over flagging, and never claims a result the model alone cannot deliver.

These three numbers are not decoration. Step 4 uses them directly to choose the probability cutoff, which is the one place in this project where cost and benefit actually meet. A cutoff picked without them is a cutoff picked by accident.

## What Step 1 Settles

Four decisions carry into the rest of the project.

1. The task is binary classification, Dropout vs Graduate, with Enrolled removed.
2. The model uses enrollment time features only, dropping all semester records to avoid leakage.
3. The model is chosen on PR AUC and ROC AUC, which hold across every cutoff. Recall is the number the school cares about, but it only means something once a cutoff is fixed, so Step 4 fixes one on purpose. Accuracy is reported and not trusted.
4. Value is students kept minus outreach cost, at 2,100 euros a student against 50 to 300 euros a contact, with stated assumptions. Step 4 turns these into the cutoff.

Step 3 acts on the leakage decision in code. Step 4 picks the cutoff using the value numbers above. Step 5 revisits the scope limit and the fairness of the flags.